In [1]:
import pandas as pd

df = pd.read_csv("data/fake_news.csv")
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20800 non-null  int64 
 1   title   20242 non-null  object
 2   author  18843 non-null  object
 3   text    20761 non-null  object
 4   label   20800 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 812.6+ KB


In [3]:
df.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [4]:
df.shape

(20800, 5)

In [5]:
# drop rows where text is null
df = df.dropna(subset=["text"])

In [6]:
df.isnull().sum()

id           0
title      558
author    1918
text         0
label        0
dtype: int64

In [7]:
df.columns

Index(['id', 'title', 'author', 'text', 'label'], dtype='object')

In [8]:
df = df.drop(["id", "title", "author"], axis=1)

In [9]:
df.head()

,text,label
0,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,Ever get the feeling your life circles the rou...,0
2,"Why the Truth Might Get You Fired October 29, ...",1
3,Videos 15 Civilians Killed In Single US Airstr...,1
4,Print \nAn Iranian woman has been sentenced to...,1


In [10]:
import nltk

nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
port_stem = PorterStemmer()

In [12]:
port_stem

<PorterStemmer>

In [13]:
port_stem.stem("My nAme Is TRUC !@#$%&*")

'my name is truc !@#$%&*'

In [14]:
import re

In [15]:
def stemming(content):
    con = re.sub("[^a-zA-Z]", " ", content)
    con = con.lower()
    con = con.split()
    con = [
        port_stem.stem(word) for word in con if not word in stopwords.words("english")
    ]
    con = " ".join(con)
    return con

In [16]:
stemming("Hello this is an orange juice")

'hello orang juic'

In [17]:
df["text"] = df["text"].apply(stemming)

In [18]:
df["label"].value_counts()

label
0    10387
1    10374
Name: count, dtype: int64

In [19]:
x = df["text"]

In [20]:
y = df["label"]

In [21]:
x.shape

(20761,)

In [22]:
y.shape

(20761,)

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20)

In [25]:
vect = TfidfVectorizer()

In [26]:
x_train = vect.fit_transform(x_train)
x_test = vect.transform(x_test)

In [27]:
x_train.shape

(16608, 98802)

In [28]:
x_test.shape

(4153, 98802)

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

#### Decision tree


In [30]:
dt_model = DecisionTreeClassifier()

In [31]:
dt_model.fit(x_train, y_train)

DecisionTreeClassifier()

In [32]:
y_pred_dt = dt_model.predict(x_test)

In [33]:
y_pred_dt

array([1, 0, 1, ..., 0, 1, 1], dtype=int64)

In [34]:
accuracy_score(y_pred_dt, y_test)

0.8791235251625331

In [35]:
dt_model.score(x_test, y_test)

0.8791235251625331

#### Random Forest


In [36]:
rf_model = RandomForestClassifier(random_state=42)

In [37]:
rf_model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [38]:
y_pred_rf = rf_model.predict(x_test)

In [39]:
accuracy_score(y_pred_rf, y_test)

0.9162051529015169

In [40]:
rf_model.score(x_test, y_test)

0.9162051529015169

#### Adaboost


In [41]:
from sklearn.ensemble import AdaBoostClassifier

adb_model = AdaBoostClassifier(n_estimators=100, random_state=42)
adb_model.fit(x_train, y_train)

c:\Users\user\anaconda3\envs\fakenews_env\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoostClassifier(n_estimators=100, random_state=42)

In [42]:
y_pred_adb = adb_model.predict(x_test)

In [43]:
accuracy_score(y_pred_adb, y_test)

0.9328196484469059

In [44]:
adb_model.score(x_test, y_test)

0.9328196484469059

#### Gradient Boosting


In [45]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

In [46]:
gb_model.fit(x_train, y_train)

GradientBoostingClassifier(random_state=42)

In [47]:
y_pred_gb = gb_model.predict(x_test)

In [48]:
accuracy_score(y_pred_gb, y_test)

0.9267999036840838

In [49]:
gb_model.score(x_test, y_test)

0.9267999036840838

#### XGBoost


In [50]:
from xgboost import XGBRFClassifier

xgb_model = XGBClassifier(n_estimators = 100)
xgb_model.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [51]:
y_pred_xgb = xgb_model.predict(x_test)

In [52]:
accuracy_score(y_pred_xgb, y_test)

0.9528052010594751

In [53]:
xgb_model.score(x_test, y_test)

0.9528052010594751

In [ ]:
# import joblib

# # Save TF-IDF vectorizer
# joblib.dump(vect, "models/tfidf_vectorizer.pkl")

# # Save trained XGBoost model
# joblib.dump(xgb_model, "models/final_model.pkl")

In [54]:
import joblib

# Save TF-IDF vectorizer and trained XGBoost model
joblib.dump((vect, xgb_model), "fake_news_model.pkl")

['fake_news_model.pkl']

In [ ]:
# vect = joblib.load("models/tfidf_vectorizer.pkl")
# xgb_model = joblib.load("models/final_model.pkl")

In [55]:
vect, xgb_model = joblib.load("fake_news_model.pkl")

In [56]:
news_1 = ["""Clinton Campaign Demands FBI Affirm Trump's Russia Ties   
With the 2016 election campaign winding down, the Clinton campaign is ratcheting up demands for the FBI to publicly confirm the campaignâ€™s allegations that Republican nominee Donald Trump is secretly in league with Russia. Sen. Harry Reid (D â€“ NV) went so far as to claim the FBI has secret â€œexplosiveâ€ evidence of coordination between the Trump campaign and the Russian government that it is withholding. 
FBI officials familiar with their investigations into the allegations, which the Clinton campaign started publicizing around the Democratic National Convention, say theyâ€™ve turned up nothing to connect Trump and Russia , leading FBI Director James Comey to decide against making any statements to that effect. 
The Clinton campaign has been making the allegations so long that they have taken to claiming â€œeveryone knowsâ€ that they are true, and appears unsettled by the FBIâ€™s refusal to sign off on the claims simply because they havenâ€™t been able to find real evidence corroborating the story. 
The Trump campaign has repeatedly denied ties to Russia, but that didnâ€™t stop Clinton from calling Trump a â€œpuppetâ€ of Russian President Vladimir Putin during the final presidential debate. The calls have grown since Fridayâ€™s FBI report to Congress about further Clinton emails being sought. 
With Clintonâ€™s main campaign scandal growing in the waning weeks of the deal, some in her campaign have suggested that affirming Trump as secretly in league with the Russians would only be fair. Absent any evidence, however, it appears that wonâ€™t be happening.  
"""]

In [57]:
# Convert text to TF-IDF features
text_vector = vect.transform(news_1)

# Predict
prediction = xgb_model.predict(text_vector)

if prediction == [0]:
    print("Reliable News")
else:
    print("Unreliable News")

Unreliable News
